In [1]:
from IPython.core.display import display, HTML
import sympy.physics
import sympy.physics.wigner
display(HTML("<style>.container { width:100% !important; }</style>"))
import sys
sys.path.append("C:/home/Python/")
sys.path.append("../")

import numpy as np                # module for vectorized numeric calculation

from spectra.Util.HelpUtil import help_

from sympy import *
import sympy as sp

def _int_or_halfint(value):
    """return Python int unless value is half-int (then return float)"""
    """added numpy.int*/float* support"""

    from sympy.core.numbers import (Float, Rational)
    import numpy

    if isinstance(value, int) or isinstance(value, numpy.integer):
        return int(value)
    elif (type(value) is float) or isinstance(value, numpy.floating):
        if value.is_integer():
            return int(value)  # an int
        if (2*value).is_integer():
            return float(value)  # a float
    elif isinstance(value, Rational):
        if value.q == 2:
            return value.p/value.q  # a float
        elif value.q == 1:
            return value.p  # an int
    elif isinstance(value, Float):
        return _int_or_halfint(float(value))
    raise ValueError("expecting integer or half-integer, got %s" % value)


import sympy
sympy.physics.wigner._int_or_halfint = _int_or_halfint  # overload sympy.physics.wigner._int_or_halfint
del sympy


C:\Users\ichim\AppData\Local\Temp\ipykernel_29952\427294188.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [2]:
# np.int8/int16/int32/int64/...   np.float16/float32/float64/...
isinstance(np.arange(-6,7)[0], np.integer), isinstance(np.arange(-6.5,7.5)[0], np.floating)

(True, True)

In [3]:
(np.arange(-6.5,7.5)[0]*2).is_integer()

True

In [4]:
## half interger
(0.5*2).is_integer()

True

In [4]:
from sympy.physics.wigner import wigner_3j

class mc_class():  # multi-component class
    # import sympy as sp    # already "imported sympy as sp" globally
    count = 0               # クラス変数    # dangerous to use class variable
                                           # reason:
                                           #   multiple mc_class instance for example mc1, mc2, ...
                                           #   will share the same "count" variable
                                           #   perhaps better to do "self.count=0" in def __init__ function
    def __init__(self,J,K,Q):     # コンストラクタ
        n = 2*J+1
        Tkq = sp.zeros(n,n) # multipole component matrix
        self.J = J          # インスタンス変数
        self.K = K          # インスタンス変数
        self.Q = Q          # 　　　　” 
        Ms = np.arange(-J,J+1)
        for i in range(0, np.size(Ms)):
            # m1 = int(Ms[i]) # np.int64からintに変換
            m1 = Ms[i]
            for j in range(0, np.size(Ms)):
                # m2 = int(Ms[j]) # np.int64からintに変換
                m2 = Ms[j]
                coef = (-1)**(J-m1) *sp.sqrt(2*K+1) * wigner_3j(J, J, K, m1, -m2, -Q)
                Tkq[i,j] = coef
        abs_non_zero = [element for element in Tkq.applyfunc(Abs) if element != 0]
        fct = min(abs_non_zero)
        self.Tkq = Tkq   #　pardial dij matrix for the moment
        self.fct = fct   # factor
        self.sym = r'\rho^{%s}_{%s}' %(K,Q)   # symbol
        self.rokq = 0   # to be replaced by evaluating formula or element value
        


    
J = 1; K = 1; Q = 1
mc1 = mc_class(J,K,Q)


help_(mc1)

------------------------------------------------------------------------------------------
name                       type                                 value/len/shape
------------------------------------------------------------------------------------------
mc_class
|- J                       int                                  v: 1
|- K                       int                                  v: 1
|- Q                       int                                  v: 1
|- Tkq                     symMat                               s: (3, 3)
|- fct                     symExpr                              v: sqrt(2)/2
|- sym                     str                                  v: \rho^{1}_{1}
|- rokq                    int                                  v: 0


In [2]:
from sympy.physics.wigner import wigner_3j
from dataclasses import dataclass, field

@dataclass
class mc_class:  # multi-component class

    J: int | float                      # input argument
    K: int | float                      # input argument
    Q: int | float                      # input argument
    

    # use field with init=False, then these instance variable will not be able 
    # to be modified by input keyword argument like mc_class(count=1).
    # to add these variable into the input argument, set init=True, or just 
    # change (for example)
    #     count: int = field(default=0, init=False) 
    # to
    #     count: int = 0
    count: int = field(default=0, init=False)                     
                                        # initialized as 0, to be modified later
    Tkq: sp.MatrixBase = field(default_factory=lambda: sp.zeros(1, 1), init=False)  
                                        # placeholder, to be modified later
    fct: sp.Expr = field(default_factory=lambda: sp.Expr(0), init=False)             
                                        # placeholder, to be modified later
    sym: str = field(default="", init=False)                        
                                        # placeholder, to be modified later
    rokq: int = field(default=0, init=False)                         
                                        # placeholder, to be modified later
                                        # to be replaced by evaluating formula or element value

    def __post_init__(self):
        
        # if not satisfied as int_or_halfint then raise Error
        J, K, Q = [_int_or_halfint(v) for v in (self.J, self.K, self.Q)]
        self.J, self.K, self.Q = J, K, Q
        n: int = int(2*J+1)
        Tkq = sp.zeros(n,n)

        Ms: list[int|float] = [-J + v for v in range(n)] # sp.Rationalも?
        assert len(Ms) == n
        for i, m1 in enumerate(Ms):
            for j, m2 in enumerate(Ms):
                coef = (-1)**(J-m1) *sp.sqrt(2*K+1) * wigner_3j(J, J, K, m1, -m2, -Q)
                Tkq[i,j] = coef
        abs_non_zero = [element for element in Tkq.applyfunc(Abs) if element != 0]
        assert len(abs_non_zero), f"empty abs_non_zero, anything wrong with the input {J=},{K=},{Q=}"
        fct = min(abs_non_zero)
        self.Tkq = Tkq   #　pardial dij matrix for the moment
        self.fct = fct   # factor
        self.sym = r'\rho^{%s}_{%s}' %(K,Q)   # symbol
        
        # already initialized
        # self.rokq = 0   # to be replaced by evaluating formula or element value


        
J = 1.5; K = 1; Q = 1
mc1 = mc_class(J,K,Q)


help_(mc1)

------------------------------------------------------------------------------------------
name                       type                                 value/len/shape
------------------------------------------------------------------------------------------
mc_class
|- J                       float                                v: 1.5
|- K                       int                                  v: 1
|- Q                       int                                  v: 1
|- Tkq                     symMat                               s: (4, 4)
|- fct                     symExpr                              v: 0.1*sqrt(30)
|- sym                     str                                  v: \rho^{1}_{1}
